# 파일럿 실행 (P0 ~ P7)
위에서부터 한 칸씩 실행하세요. **런타임 유형: T4 GPU**. 결과는 모두 `내 드라이브/hn/runs/`에 저장됩니다.
코랩이 끊기면 1번 셀부터 다시 실행하면 됩니다. 끝난 실행(DONE)은 자동으로 건너뜁니다.

## 1. 드라이브 연결 + 코드 불러오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/hn/runs', exist_ok=True)

%cd /content
if not os.path.exists('hard-negative-mllm'):
    !git clone -q https://github.com/naha0106-tech/hard-negative-mllm.git
%cd /content/hard-negative-mllm
!git pull -q
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. 설치 + 버전 기록 + 모델 버전 고정
- 라이브러리 버전은 `hn/env_pilot.txt`에 저장됩니다.
- 모델 revision은 **처음 한 번만** 최신 해시를 읽어 `hn/model_revision.txt`에 저장하고, 이후 세션은 그 값을 그대로 씁니다.

In [ ]:
!pip install -q transformers==4.51.3 peft==0.15.2 accelerate datasets pyarrow pyyaml
!pip freeze | grep -E "^(torch|transformers|peft|accelerate|datasets)==" | tee /content/drive/MyDrive/hn/env_pilot.txt

import yaml
from huggingface_hub import HfApi
REV_FILE = '/content/drive/MyDrive/hn/model_revision.txt'
cfg = yaml.safe_load(open('config.yaml'))
model_id = cfg['models']['smolvlm']['id']
if os.path.exists(REV_FILE):
    rev = open(REV_FILE).read().strip()
else:
    rev = cfg['models']['smolvlm'].get('revision') or HfApi().model_info(model_id).sha
    open(REV_FILE, 'w').write(rev)
text = open('config.yaml', encoding='utf-8').read()
text = text.replace('    revision: null\n    longest_edge', f'    revision: {rev}\n    longest_edge', 1)
open('config.yaml', 'w', encoding='utf-8').write(text)
print('SmolVLM revision =', yaml.safe_load(open('config.yaml'))['models']['smolvlm']['revision'])

## 3. P0 — 학습 전 점검 C1~C8
전부 `통과`가 나와야 다음으로 갑니다. 첫 실행은 데이터셋·모델 다운로드 때문에 오래 걸립니다.

In [ ]:
!python scripts/04a_pilot_checks.py

## 4. P1~P3 — 학습률 3개 비교 (Rand 400문항)

In [ ]:
import yaml, sys
sys.path.insert(0, 'scripts')
cfg = yaml.safe_load(open('config.yaml'))
runs = cfg['pilot']['runs']
from train_common import pilot_run_id
for key in ['P1', 'P2', 'P3']:
    spec = runs[key]
    rid = pilot_run_id(key, spec, spec['lr'])
    !python scripts/04_train.py --trainset {spec['trainset']} --run_id {rid} --lr {spec['lr']} --seed 0

## 5. 학습률 고르기 (B2)

In [ ]:
!python scripts/04b_pilot_report.py --stage lr
import json
LR = json.load(open('/content/drive/MyDrive/hn/runs/pilot_lr.json'))['chosen_lr']
print('채택 학습률 =', LR)

## 6. P4~P7 — 채택 학습률로 1,200문항 학습
P4 Rand, P5 Hard, P6 Easy, P7 Rand 반복

In [ ]:
import json
LR = json.load(open('/content/drive/MyDrive/hn/runs/pilot_lr.json'))['chosen_lr']
for key in ['P4', 'P5', 'P6', 'P7']:
    spec = runs[key]
    rid = pilot_run_id(key, spec, LR)
    !python scripts/04_train.py --trainset {spec['trainset']} --run_id {rid} --lr {LR} --seed 0

## 7. 관문 B 성적표 (B1~B6)
결과는 `hn/runs/pilot_report.json`에 저장됩니다.

In [ ]:
!python scripts/04b_pilot_report.py

## 8. (참고) 실행별 요약 한눈에 보기

In [ ]:
import pandas as pd, glob, json, os
rows = []
for p in sorted(glob.glob('/content/drive/MyDrive/hn/runs/pilot_*/summary.json')):
    s = json.load(open(p)); s['run'] = os.path.basename(os.path.dirname(p)); rows.append(s)
pd.DataFrame(rows).set_index('run')